In [1]:
import os
from pathlib import Path

import polars as pl
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import mllabs 
mllabs.__version__

from mllabs.processor import PolarsLoader, PandasConverter, ExprProcessor
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

data_path = Path('data')

ploader = PolarsLoader(predefined_types={'id': pl.Int64, 'PitNextLap': pl.Float32})
ploader.fit([data_path / 'train.csv', data_path / 'test.csv', data_path / 'f1_strategy_dataset_v4.csv'])

pconv = PandasConverter(index_col= 'id')
expr_dict = {
    'Position_Change': pl.col('Position_Change').cast(pl.Int8),
}
expr_proc = ExprProcessor(expr_dict)

df_train = pconv.fit_transform(
    expr_proc.fit_transform(ploader.transform([data_path / 'train.csv']))
)
loader = make_pipeline(ploader, expr_proc, pconv)
df_test = loader.transform([data_path / 'test.csv'])
df_train['PitNextLap'] = df_train['PitNextLap'].astype('int8')

2026-06-01 23:59:27.913654: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-01 23:59:28.242403: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-01 23:59:29.487284: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/sun9sun9/python312/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: Fu

In [2]:
X_bin = ['PitStop']
X_nom = ['Driver', 'Compound', 'Race', 'Year']
X_int = ['LapNumber', 'Stint', 'Position', 'Position_Change']
X_cont = ['TyreLife', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress']
y = 'PitNextLap'

s_lap_progress = pd.concat([
    df_train.loc[(df_train['LapNumber'] == 1) & (df_train['TyreLife'] == 1.0)], 
    df_test.loc[(df_test['LapNumber'] == 1) & (df_test['TyreLife'] == 1.0)]
]).groupby('Race')['RaceProgress'].apply(lambda x: x.mode()[0])
s_total_lap = (1 / s_lap_progress).round()
df_drv = pd.concat([
    df_train[['Driver', 'Position']], df_test[['Driver', 'Position']]
])
df_drv_stat = df_drv.groupby('Driver')['Position'].agg(['mean', 'count']).rename(columns={'mean': 'Drv_Position_mean', 'count': 'Drv_count'})

In [3]:
def process(df):
    return df.join(df_drv_stat, on='Driver').assign(
        Total_Lap = lambda x: x['Race'].map(s_total_lap),
        TyreLife2 = lambda x: x['TyreLife'] / x['Total_Lap'],
        TyreLife3 = lambda x: x['RaceProgress'] / x['LapNumber'] * x['TyreLife'],
        diff = lambda x: x['LapNumber'] - x['TyreLife'],
        RaceProgress2 = lambda x: x['Race'].map(s_lap_progress) * x['LapNumber'],
        diff2 = lambda x: x['RaceProgress2'] - x['RaceProgress'],
        LapNumber2 = lambda x: x['RaceProgress'] * x['Total_Lap'],
        diff3 = lambda x: x['LapNumber'] - x['LapNumber2'],
        diff_Position = lambda x: x['Position'] - x['Drv_Position_mean'],
        Year_c = lambda x: x['Year'].astype('str').astype('category'),
        Total_Lap_c = lambda x: x['Total_Lap'].astype('str').astype('category')
    )
df_train = process(df_train)
df_test = process(df_test)

In [4]:
df_tmp = pd.concat([
    df_train[['Compound', 'TyreLife3']],
    df_test[['Compound', 'TyreLife3']]
], axis=0).groupby('Compound')['TyreLife3'].transform(lambda x: (x - x.mean()) / x.std())
df_train['TyreLife3_Compound_norm'] = df_tmp.loc[df_train.index]
df_test['TyreLife3_Compound_norm'] = df_tmp.loc[df_test.index]

q_val = pd.concat([df_train['RaceProgress'], df_test['RaceProgress']]).quantile(q = np.arange(0, 21) / 20)
df_train['RaceProgress_q1'] = pd.cut(df_train['RaceProgress'], q_val)
df_test['RaceProgress_q1'] = pd.cut(df_test['RaceProgress'], q_val)

df_timedelta_std = pd.concat([
    df_train[['RaceProgress_q1', 'LapTime_Delta', 'Cumulative_Degradation']],
    df_test[['RaceProgress_q1', 'LapTime_Delta', 'Cumulative_Degradation']]
]).groupby(['RaceProgress_q1'])[['LapTime_Delta', 'Cumulative_Degradation']].transform(
    lambda x: (x - x.mean()) / x.std()).rename(columns = lambda x: x + '_std')
df_train = df_train.join(df_timedelta_std)
df_test = df_test.join(df_timedelta_std)

df_train['Stint2'] = df_train['Stint'].clip(1, 5)
df_test['Stint2'] = df_train['Stint'].clip(1, 5)

s_mean_tyrelife = pd.concat([
    df_train.loc[df_train['PitStop'] == 1],
    df_test.loc[df_test['PitStop'] == 1]
]).groupby(
    ['Compound', 'Stint2']
)[['TyreLife2', 'TyreLife3', 'Cumulative_Degradation', 'LapTime_Delta']].agg(lambda x: x.clip(*x.quantile([0.01, 0.99])).mean()).rename(columns = lambda x: x + '_mean')

df_train = df_train.join(s_mean_tyrelife, on = ['Compound', 'Stint2'])
df_train['TyreLife_diff'] = df_train['TyreLife3'] - df_train['TyreLife3_mean']
df_train['Cumulative_Degradation_diff'] = df_train['Cumulative_Degradation'] - df_train['Cumulative_Degradation_mean']
df_train['LapTime_Delta_diff'] = df_train['LapTime_Delta'] - df_train['LapTime_Delta_mean']
df_train['RaceProgress3'] = df_train['RaceProgress2'].clip(0, 1)

df_test = df_test.join(s_mean_tyrelife, on = ['Compound', 'Stint2'])
df_test['TyreLife_diff'] = df_test['TyreLife3'] - df_test['TyreLife3_mean']
df_test['Cumulative_Degradation_diff'] = df_test['Cumulative_Degradation'] - df_test['Cumulative_Degradation_mean']
df_test['LapTime_Delta_diff'] = df_test['LapTime_Delta'] - df_test['LapTime_Delta_mean']
df_test['RaceProgress3'] = df_test['RaceProgress2'].clip(0, 1)

In [5]:
X_bin = ['PitStop']
X_nom = ['Compound', 'Race', 'Year_c', 'Total_Lap_c']
X_int = ['LapNumber', 'Stint', 'Position', 'Position_Change']
X_cont = (['TyreLife', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress'] + 
          ['LapNumber2', 'RaceProgress3', 'Total_Lap', 'TyreLife2', 'TyreLife3', 'diff', 'diff2', 'diff3', 'diff_Position'] + 
          ['TyreLife_diff', 'Cumulative_Degradation_diff', 'LapTime_Delta_diff', 'TyreLife3_Compound_norm'])
y = 'PitNextLap'

In [12]:
from sklearn.model_selection import StratifiedKFold
from mllabs import Experimenter, Connector, ColSelector, ProgressSessionLogger, TqdmProgressSession
from mllabs.adapter import XGBoostAdapter, LightGBMAdapter, CatBoostAdapter
from mllabs.nn import NNClassifier
from mllabs.col import ohe_drop_first
from mllabs.collector import SHAPCollector, StackingCollector, ProbToLabel
from mllabs.filter import RandomFilter
from mllabs.processor import PolarsLoader, PandasConverter, TypeConverter, CatConverter, ExprProcessor, CrossFitTransformer
from mllabs.collector import MetricCollector, ModelAttrCollector

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

skf1 = StratifiedKFold(5, random_state = 123, shuffle=True)

In [7]:
if not os.path.exists('exp/modeling'):
    ex = Experimenter.create(
        df_train, 'exp/modeling', sp = skf1, splitter_params={'y': y}, 
        logger=ProgressSessionLogger(level=['info', 'progress'], session_cls=TqdmProgressSession))
    ex.set_grp('clf', method = 'predict_proba', role = 'head', edges = {'y': [(None, y)]})
    ex.set_grp(
        'xgb', parent = 'clf', processor=xgb.XGBClassifier, adapter=XGBoostAdapter(eval_mode='both'), 
        params={'random_state': 123, 'n_estimators': 1000, 'learning_rate': 0.05, 'enable_categorical': True})
    ex.set_grp(
        'lgb', parent = 'clf', processor=lgb.LGBMClassifier, adapter=LightGBMAdapter(eval_mode='both'), 
        params={'random_state': 123, 'n_estimators': 1000, 'learning_rate': 0.05, 'verbose': -1})
    ex.set_grp(
        'cb', parent = 'clf', processor=cb.CatBoostClassifier, adapter=CatBoostAdapter(eval_mode='valid'),
        params={'cat_features': ColSelector(col_type='category')})
    ex.set_grp('nn', parent = 'clf', processor = NNClassifier, params = {'metrics': ['sparse_categorical_crossentropy']})
    ex.set_grp('pre', role = 'stage', method='transform')
    ex.set_grp('pre_ft', role = 'stage', method='fit_transform', edges = {'y': [(None, y)]})
    ex.build()
    
    y_edges = {'y': [(None, y)]}
    ex.add_collector(
        MetricCollector('roc', Connector(edges = y_edges, role = 'head'), slice(-1, None), roc_auc_score, include_train = True))
    ex.add_collector(
        StackingCollector(
            'stacking', Connector(edges=y_edges, role='head'),
            slice(-1, None), method='mean'
        ))
else:
    ex = Experimenter.load('exp/modeling', df_train, logger=ProgressSessionLogger(level=['info', 'progress'], session_cls=TqdmProgressSession))

Loaded: 2 node(s), 7 group(s), 5 fold(s)


In [8]:
ex.set_node('xgb_base', grp='xgb', edges={'X': [(None, ['^' + i + '$' for i in X_bin + X_nom + X_int + X_cont])]}, 
            params={'n_estimators': 3000, 'gpu': 'yes', 'max_depth': 6, 'learning_rate': 0.03, 'colsample_bytree': 0.75, 'subsample': 0.75, 'min_child_weight': 32})
ex.set_node('lgb_base', grp='lgb', edges={'X': [(None, ['^' + i + '$' for i in X_bin + X_nom + X_int + X_cont])]}, 
            params={'n_estimators': 3000, 'num_leaves': 31, 'learning_rate': 0.03, 'colsample_bytree': 0.75, 'subsample': 0.75, 'min_child_weight': 64})
ex.exp(finalize=True, n_jobs=1, gpu_id_list=[])

No head nodes to experiment


In [9]:
ex.add_trainer('trainer')
ex.trainers['trainer'].select_head(['xgb_base', 'lgb_base'])
ex.trainers['trainer'].train(n_jobs=2, gpu_id_list=[0])

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

2026-06-02 00:00:20.854754: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-02 00:00:20.854754: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-02 00:00:20.893188: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-02 00:00:20.893320: I tensorflow/core/platform/cpu_feature_guard.cc:210] This Tenso

Train complete: 2 node(s)


In [10]:
df_stacking = ex.get_collector('stacking').get_dataset()
df_stacking

,lgb_base__PitNextLap_1,xgb_base__PitNextLap_1,PitNextLap
id,,,
2,0.710431,0.691426,1
3,0.000651,0.000299,0
4,0.264167,0.252417,0
37,0.281092,0.285127,0
43,0.770158,0.753383,1
...,...,...,...
439127,0.000428,0.000564,0
439129,0.000490,0.000533,0
439133,0.000558,0.000662,0


In [14]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

clf_lr = LogisticRegression(C = 1000)
X_stk = [i for i in df_stacking.columns if i != y]
scores_stk = cross_val_score(clf_lr, df_stacking[X_stk], df_stacking[y], scoring = 'roc_auc', cv = skf1)
scores_stk.mean()

np.float64(0.9505700268617128)

In [15]:
clf_lr.fit(df_stacking[X_stk], df_stacking[y])

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1000
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mu

In [66]:
# 지금 사용하는 XGBoost에서 Categorical 변를 제대로 처리 못하는 버그가 있어 'lgb_base'로만 
infer = ex.trainers['trainer'].to_inferencer(slice(-1, None))
prd = pd.Series(
    infer.process(df_test, nodes=['lgb_base']).iloc[:, 0], index = pd.Index(df_test.index, name = 'id'), name = y
)
prd.to_csv('data/Submission.csv')
# !kaggle competitions submit -c playground-series-s6e5 -f data/Submission.csv -m "1"